# Task 3 — Jaccard Results Table
**Team Member 3: Generalization Specialist**

Reads the output of `olmoe-jaccard-diagnostic.ipynb` and formats the results
in the same comparison table used for Phi-3.5-MoE in the thesis.

**Run `olmoe-jaccard-diagnostic.ipynb` first** — this notebook requires
`olmoe_experiment_config.json` to exist in the current directory.

## What this produces
1. Side-by-side comparison table: Phi-3.5-MoE vs OLMoE Jaccard scores
2. Per-layer table for OLMoE with top-3 highest-overlap layers highlighted
3. Interpretation note for the thesis

In [1]:
import json
import numpy as np
import os

# On Kaggle: set this to the path where olmoe_experiment_config.json was saved
# Usually "/kaggle/working/olmoe_experiment_config.json" or upload it as a dataset input
CONFIG_PATH = "/kaggle/input/datasets/ruhanisnot/olmoe-jaccard-test/olmoe-jaccard-test/olmoe_experiment_config.json"

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(CONFIG_PATH + " not found. Run olmoe-jaccard-diagnostic.ipynb first.")

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

print("Config loaded:")
print(json.dumps(cfg, indent=2))


Config loaded:
{
  "model_name": "allenai/OLMoE-1B-7B-0125",
  "model_dim": 2048,
  "ffn_dim": 1024,
  "num_layers": 16,
  "num_experts": 64,
  "top_k": 8,
  "jaccard_python_medical": 0.1029,
  "jaccard_math_creative": 0.0528,
  "jaccard_null_baseline": 0.8403,
  "top_n_jaccard": 16,
  "chosen_option": "B",
  "chosen_pair": "Math vs Creative",
  "target_layer": 6,
  "target_expert": 18,
  "primary_dataset": "DigitalLearningGmbH/MATH-lighteval",
  "primary_col": "problem",
  "conflict_dataset": "roneneldan/TinyStories",
  "conflict_col": "text",
  "conflict_config": null,
  "domain_names": [
    "math",
    "creative"
  ]
}


## Table 1 — Model Comparison (matches Phi table format from thesis)

In [2]:
PHI_JACCARD_A = 0.056
PHI_JACCARD_B = 0.069

olmoe_jaccard_a = cfg["jaccard_python_medical"]
olmoe_jaccard_b = cfg["jaccard_math_creative"]
olmoe_null      = cfg["jaccard_null_baseline"]
olmoe_top_n     = cfg.get("top_n_jaccard", 16)

sep_ratio_a = olmoe_jaccard_a / max(olmoe_null, 0.001)
sep_ratio_b = olmoe_jaccard_b / max(olmoe_null, 0.001)

def olmoe_status(score, null):
    ratio = score / max(null, 0.001)
    if ratio < 0.2:   return "Separated (ratio=%.2fx)" % ratio
    elif ratio < 0.5: return "Moderate overlap (ratio=%.2fx)" % ratio
    else:             return "High overlap (ratio=%.2fx)" % ratio

rows = [
    ("Domain Pair",        "Phi-3.5-MoE",   "OLMoE-1B-7B",  "Status (OLMoE)"),
    ("-"*28,               "-"*12,           "-"*12,          "-"*24),
    ("Python vs Medical",  "%.3f" % PHI_JACCARD_A, "%.3f" % olmoe_jaccard_a, olmoe_status(olmoe_jaccard_a, olmoe_null)),
    ("Math vs Creative",   "%.3f" % PHI_JACCARD_B, "%.3f" % olmoe_jaccard_b, olmoe_status(olmoe_jaccard_b, olmoe_null)),
    ("Null (same domain)", "~1.0",           "%.3f" % olmoe_null, "baseline"),
]
for r in rows:
    print("%-30s %14s %14s  %s" % r)

print()
arch_rows = [
    ("Architecture",         "Phi-3.5-MoE",  "OLMoE-1B-7B"),
    ("-"*28,                 "-"*12,          "-"*12),
    ("Experts per layer",    "16",            "64"),
    ("Top-k routing",        "2",             "8"),
    ("Uniform baseline",     "12.5%",         "12.5%"),
    ("Jaccard method",       "raw set",       "top-%d" % olmoe_top_n),
]
for r in arch_rows:
    print("%-30s %14s %14s" % r)

chosen_j     = olmoe_jaccard_b if cfg["chosen_option"] == "B" else olmoe_jaccard_a
chosen_ratio = chosen_j / max(olmoe_null, 0.001)
print()
print("Chosen pair: %s (Option %s)" % (cfg["chosen_pair"], cfg["chosen_option"]))
print("  Top-K Jaccard:    %.3f" % chosen_j)
print("  Null baseline:    %.3f" % olmoe_null)
print("  Separation ratio: %.2fx above null" % chosen_ratio)


Domain Pair                       Phi-3.5-MoE    OLMoE-1B-7B  Status (OLMoE)
----------------------------     ------------   ------------  ------------------------
Python vs Medical                       0.056          0.103  Separated (ratio=0.12x)
Math vs Creative                        0.069          0.053  Separated (ratio=0.06x)
Null (same domain)                       ~1.0          0.840  baseline

Architecture                      Phi-3.5-MoE    OLMoE-1B-7B
----------------------------     ------------   ------------
Experts per layer                          16             64
Top-k routing                               2              8
Uniform baseline                        12.5%          12.5%
Jaccard method                        raw set         top-16

Chosen pair: Math vs Creative (Option B)
  Top-K Jaccard:    0.053
  Null baseline:    0.840
  Separation ratio: 0.06x above null


## Table 2 — Per-Layer Jaccard for OLMoE (Top-3 Highest-Overlap Layers)

The diagnostic notebook saves per-layer data in the config. If per-layer arrays
are not in the config, we reconstruct them from the saved frequency pickles
(if available) or note that the diagnostic must be re-run with per-layer saving.

**Note:** The diagnostic Cell 11 prints per-layer scores to stdout — check
that notebook's output for the full per-layer breakdown.

In [3]:
import pickle

# Update FREQ_CACHE path the same way as CONFIG_PATH above
FREQ_CACHE = "/kaggle/input/datasets/ruhanisnot/olmoe-jaccard-test/olmoe-jaccard-test/olmoe_freq_cache.pkl"

if os.path.exists(FREQ_CACHE):
    print(f"Loading frequency cache from {FREQ_CACHE}...")
    with open(FREQ_CACHE, "rb") as f:
        freq_data = pickle.load(f)
    # Load the chosen pair's frequency data (not always Python/Medical)
    chosen = cfg["chosen_option"]
    if chosen == "B":
        freq_a = freq_data["math"]
        freq_b = freq_data["creative"]
        label_a, label_b = "Math", "Creative"
    else:
        freq_a = freq_data["python"]
        freq_b = freq_data["medical"]
        label_a, label_b = "Python", "Medical"
    has_freq = True
    print(f"Loaded: {label_a} vs {label_b} (chosen pair: {cfg['chosen_pair']})")
else:
    print(f"Frequency cache not found at {FREQ_CACHE}.")
    print("Per-layer table will be skipped.")
    has_freq = False
    freq_a = freq_b = None


Loading frequency cache from /kaggle/input/datasets/ruhanisnot/olmoe-jaccard-test/olmoe-jaccard-test/olmoe_freq_cache.pkl...
Loaded: Math vs Creative (chosen pair: Math vs Creative)


In [4]:
N_LAYERS  = cfg["num_layers"]
N_EXPERTS = cfg["num_experts"]
TOP_N     = cfg.get("top_n_jaccard", 16)

if has_freq:
    def jaccard_topk(fa, fb, top_n=TOP_N):
        top_a = set(sorted(fa, key=fa.get, reverse=True)[:top_n])
        top_b = set(sorted(fb, key=fb.get, reverse=True)[:top_n])
        union = top_a | top_b
        return len(top_a & top_b) / len(union) if union else 0.0

    topk_scores = [jaccard_topk(freq_a[i], freq_b[i]) for i in range(N_LAYERS)]
    ranked = sorted(enumerate(topk_scores), key=lambda x: x[1], reverse=True)
    top3   = ranked[:3]
    top3_layers = {l for l, _ in top3}

    print(f"TABLE 2: OLMoE Top-K Jaccard by Layer ({cfg['chosen_pair']})")
    print("-" * 60)
    print(f"  {'Layer':<10} {'Jaccard':>10}  Notes")
    print("-" * 60)
    for layer_idx, score in enumerate(topk_scores):
        marker = "  <- TOP-3" if layer_idx in top3_layers else ""
        if layer_idx == cfg["target_layer"]:
            marker += "  * TARGET"
        print(f"  Layer {layer_idx:<4} {score:>10.4f}{marker}")
    print("-" * 60)
    print(f"  Mean (all layers): {sum(topk_scores)/len(topk_scores):.4f}")
    print()
    print("Top-3 highest-overlap layers:")
    for rank, (layer_idx, score) in enumerate(top3, 1):
        print(f"  #{rank}: Layer {layer_idx} -- Jaccard = {score:.4f}")
    print()
    print(f"Target layer: {cfg['target_layer']} | target expert: {cfg['target_expert']}")
else:
    print("Frequency cache not available. To enable this table:")
    print("  1. Re-run olmoe-jaccard-diagnostic.ipynb")
    print("  2. Download olmoe_freq_cache.pkl and upload to Kaggle")
    print()
    print(f"Target layer (from config): {cfg['target_layer']}")
    print(f"Target expert (from config): {cfg['target_expert']}")


TABLE 2: OLMoE Top-K Jaccard by Layer (Math vs Creative)
------------------------------------------------------------
  Layer         Jaccard  Notes
------------------------------------------------------------
  Layer 0        0.1034  <- TOP-3
  Layer 1        0.0667
  Layer 2        0.0323
  Layer 3        0.1034  <- TOP-3
  Layer 4        0.0323
  Layer 5        0.0667
  Layer 6        0.1429  <- TOP-3  * TARGET
  Layer 7        0.0667
  Layer 8        0.0323
  Layer 9        0.0323
  Layer 10       0.0667
  Layer 11       0.0000
  Layer 12       0.0323
  Layer 13       0.0667
  Layer 14       0.0000
  Layer 15       0.0000
------------------------------------------------------------
  Mean (all layers): 0.0528

Top-3 highest-overlap layers:
  #1: Layer 6 -- Jaccard = 0.1429
  #2: Layer 0 -- Jaccard = 0.1034
  #3: Layer 3 -- Jaccard = 0.1034

Target layer: 6 | target expert: 18


## Thesis Interpretation Note

In [5]:
olmoe_chosen_j = cfg["jaccard_python_medical"] if cfg["chosen_option"] == "A" else cfg["jaccard_math_creative"]
sep_ratio = olmoe_chosen_j / max(olmoe_null, 0.001)

if olmoe_chosen_j < 0.10:
    interpretation = (
        f"OLMoE's routing is even more domain-separated than Phi-3.5-MoE "
        f"(concentrated Jaccard {olmoe_chosen_j:.3f} vs Phi's 0.056). "
        "This confirms that expert-routing alone does not explain gradient entanglement — "
        "the conflict is gradient-space, not routing-space, and the Hierarchical ReLU-LoRA "
        "spawning mechanism generalises cleanly to a different MoE architecture."
    )
elif olmoe_chosen_j < 0.20:
    interpretation = (
        f"OLMoE shows low routing overlap (concentrated Jaccard {olmoe_chosen_j:.3f}), "
        f"consistent with Phi-3.5-MoE (0.056), though slightly higher due to OLMoE's "
        f"broader top-8 routing. The null baseline of {olmoe_null:.3f} confirms this is "
        f"genuine domain separation ({sep_ratio:.1f}× above chance). "
        "Gradient entanglement arises in shared parameter space, not in routing assignment, "
        "validating the spawning mechanism on this architecture."
    )
else:
    interpretation = (
        f"OLMoE shows moderate routing overlap (concentrated Jaccard {olmoe_chosen_j:.3f}), "
        "higher than Phi-3.5-MoE's 0.056. This indicates more routing-level conflict between "
        "Python and Medical domains in OLMoE's broader top-8 routing scheme. "
        "This strengthens the motivation for hierarchical spawning — both routing-space and "
        "gradient-space conflict are present, making domain interference more severe and "
        "the benefit of independent sub-adapters more pronounced."
    )

print("THESIS NOTE (copy into Section 4.2):")
print("-" * 70)
print(interpretation)
print("-" * 70)
print()
print(f"Target layer for OLMoE experiments: Layer {cfg['target_layer']}")
print(f"Target expert:                       Expert {cfg['target_expert']}")
print()
print("Pass these to the smoke test notebook (olmoe-smoke-test.ipynb):")
print(f"  TARGET_LAYER  = {cfg['target_layer']}")
print(f"  TARGET_EXPERT = {cfg['target_expert']}")

THESIS NOTE (copy into Section 4.2):
----------------------------------------------------------------------
OLMoE's routing is even more domain-separated than Phi-3.5-MoE (concentrated Jaccard 0.053 vs Phi's 0.056). This confirms that expert-routing alone does not explain gradient entanglement — the conflict is gradient-space, not routing-space, and the Hierarchical ReLU-LoRA spawning mechanism generalises cleanly to a different MoE architecture.
----------------------------------------------------------------------

Target layer for OLMoE experiments: Layer 6
Target expert:                       Expert 18

Pass these to the smoke test notebook (olmoe-smoke-test.ipynb):
  TARGET_LAYER  = 6
  TARGET_EXPERT = 18
